In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


In [5]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Корень проекта:", PROJECT_ROOT) 

Корень проекта: C:\Users\User\Desktop\face-recognition


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [7]:
class FRDataset(Dataset):
    def __init__(self, metadata_file, images_dir, transform=None):
        super().__init__()

        self.metadata = pd.read_csv(metadata_file)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        image_path = self.images_dir / row["image_id"]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["class_id"])

        return image, label




In [9]:
class FRModel(nn.Module):
    """
    Модель распознавания лиц на базе ResNet18
    с заменой последнего слоя.
    """

    def __init__(self, num_classes):
        super(FRModel, self).__init__()

        # ResNet18, предобученная на ImageNet
        self.model = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1
        )

        # Заменяем последний классификационный слой
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(
                in_features=512,
                out_features=num_classes
            )
        )

    def forward(self, x):
        return self.model(x)




In [11]:
num_classes = 400

model_ce = FRModel(num_classes=num_classes)

model_ce.eval()

x = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    outputs = model_ce(x)

print("Вход:", x.shape)
print("Выход:", outputs.shape)

assert outputs.shape == (1, 400)



Вход: torch.Size([1, 3, 224, 224])
Выход: torch.Size([1, 400])


In [12]:
# Загружаем список обучающих фотографий из второго этапа
splits_dir = PROJECT_ROOT / "data" / "processed" / "splits"

train_people_df = pd.read_csv(splits_dir / "train.csv")

print("Фотографий:", len(train_people_df))
print("Уникальных людей:", train_people_df["identity_id"].nunique())
print("Колонки:", train_people_df.columns.tolist())

Фотографий: 7983
Уникальных людей: 400
Колонки: ['image_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young', 'identity_id', 'stratum', 'x_1', 'y_1', 'width', 'height', 'lefteye_x', 'lefteye_y', 'righteye_x', 'righteye_y', 'nose_x', 'nose_y', 'leftmouth_x', 'leftmouth_y', 'rightmouth_x', 'rightmouth_y']


In [ ]:
# Сортируем идентификаторы людей
identities = sorted(train_people_df["identity_id"].unique())

# Создаём соответствие identity_id и class_id
identity_to_class = {
    identity: index
    for index, identity in enumerate(identities)
}

train_people_df["class_id"] = (
    train_people_df["identity_id"].map(identity_to_class)
)

print("Количество классов:", len(identity_to_class))
print(
    train_people_df[
        ["image_id", "identity_id", "class_id"]
    ].head()
)

assert len(identity_to_class) == 400
assert train_people_df["class_id"].between(0, 399).all()

Количество классов: 400
     image_id  identity_id  class_id
0  115262.jpg         2820       391
1  056764.jpg         2820       391
2  008286.jpg         2820       391
3  157826.jpg         2820       391
4  018909.jpg         2820       391


In [14]:
rng = np.random.default_rng(42)

train_parts = []
val_parts = []
test_parts = []

for class_id, group in train_people_df.groupby("class_id"):
    indices = rng.permutation(len(group))
    group = group.iloc[indices]

    n = len(group)

    n_train = int(n * 0.7)
    n_val = int(n * 0.15)

    train_parts.append(group.iloc[:n_train])
    val_parts.append(group.iloc[n_train:n_train + n_val])
    test_parts.append(group.iloc[n_train + n_val:])

fr_train_df = pd.concat(train_parts, ignore_index=True)
fr_val_df = pd.concat(val_parts, ignore_index=True)
fr_test_df = pd.concat(test_parts, ignore_index=True)

print("Train:", len(fr_train_df))
print("Validation:", len(fr_val_df))
print("Test:", len(fr_test_df))

print("Классов в Train:", fr_train_df["class_id"].nunique())
print("Классов в Val:", fr_val_df["class_id"].nunique())
print("Классов в Test:", fr_test_df["class_id"].nunique())

Train: 5583
Validation: 1187
Test: 1213
Классов в Train: 400
Классов в Val: 400
Классов в Test: 400


In [15]:
fr_splits_dir = PROJECT_ROOT / "data" / "processed" / "fr_splits"
fr_splits_dir.mkdir(parents=True, exist_ok=True)

fr_train_df.to_csv(fr_splits_dir / "train.csv", index=False)
fr_val_df.to_csv(fr_splits_dir / "val.csv", index=False)
fr_test_df.to_csv(fr_splits_dir / "test.csv", index=False)

print("CSV сохранены в:", fr_splits_dir)

CSV сохранены в: C:\Users\User\Desktop\face-recognition\data\processed\fr_splits
